## 1. Setup & Load

In [9]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Path to the raw dataset (adjust if your file lives elsewhere)
DATA_PATH = 'steam_reviews 1.csv'

df_raw = pd.read_csv(DATA_PATH)


## 2. Basic cleaning

In [10]:
df = df_raw.copy()

# Parse dates (raw column kept for traceability)
df['date'] = pd.to_datetime(df['date_posted'], errors='coerce')
assert df['date'].isnull().sum() == 0, "Unparseable dates found — inspect before proceeding"

# Normalise game titles: strip trademark symbols and surrounding whitespace so titles
# join/display cleanly (e.g. 'Rocket League®' -> 'Rocket League')
df['title_clean'] = (df['title']
                     .str.replace(r'[\u00AE\u2122\u00A9]', '', regex=True)  # ® ™ ©
                     .str.strip())

# Review text as string; preserve missingness info in a flag (Section 4) before filling
df['review_text'] = df['review'].fillna('').astype(str)

print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Titles normalised: {sorted(set(df['title']) - set(df['title_clean']))}")

Date range: 2010-12-20 -> 2019-02-16
Titles normalised: ['ACE COMBAT™ 7: SKIES UNKNOWN', 'Rocket League®']


## 3. Quality Flags

Each flag marks a property that *some* downstream stage will care about. No rows are removed.

| Flag | True means | 
|---|---|
| `flag_null_review` | review was missing in the raw data |
| `flag_no_words` | no word of 2+ letters in **any script** (emoji/punctuation-only, incl. Steam's `♥♥♥♥` profanity masking) |
| `flag_ascii_art` | **symbol-dominated** text: art patterns (block chars / 9+ symbol runs, excluding emphasis punctuation) **and** <30% letters |
| `flag_non_english` | >30% non-ASCII characters (heuristic for non-English text) |
| `flag_zero_hours` | reviewer had 0 recorded hours |
| `flag_extreme_hours` | hours above the 99.9th percentile |
| `flag_truncated` | review text at the 8,000-character export cap |


In [11]:
s = df['review_text']

df['flag_null_review'] = df['review'].isnull()

df['flag_no_words'] = ~s.str.contains(r'[^\W\d_]{2,}', regex=True)

has_art_pattern = (s.str.contains(r'[\u2580-\u259F]', regex=True) |
                   s.str.contains(r'''(?:([^\w\s!?.,'"])\1{8,})''', regex=True))
alpha_ratio = s.apply(lambda x: sum(c.isalpha() for c in x) / max(len(x), 1))
df['flag_ascii_art'] = has_art_pattern & (alpha_ratio < 0.30)

non_ascii_ratio = s.apply(lambda x: sum(ord(c) > 127 for c in x) / max(len(x), 1))
df['flag_non_english'] = non_ascii_ratio > 0.30

df['flag_zero_hours'] = df['hour_played'] == 0

p999 = df['hour_played'].quantile(0.999)
df['flag_extreme_hours'] = df['hour_played'] > p999

df['flag_truncated'] = s.str.len() >= 8000

flag_cols = [c for c in df.columns if c.startswith('flag_')]
print("Flag prevalence:")
print(df[flag_cols].sum().sort_values(ascending=False))
print(f"\n99.9th percentile of hours (extreme threshold): {p999:,.0f}h")

C:\Users\jaish\AppData\Local\Temp\ipykernel_14376\561021470.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  s.str.contains(r'''(?:([^\w\s!?.,'"])\1{8,})''', regex=True))


Flag prevalence:
flag_zero_hours       337
flag_no_words         108
flag_non_english       32
flag_extreme_hours     15
flag_null_review        7
flag_ascii_art          4
flag_truncated          1
dtype: int64

99.9th percentile of hours (extreme threshold): 4,999h


### Zero-hour reviews: noise or signal?

Before writing these off as junk, check how they vote — if they were random noise their
recommendation split should resemble the overall ~71% positive base rate.

In [12]:
zero = df[df['flag_zero_hours']]
print(f"Zero-hour reviews: {len(zero)} ({len(zero)/len(df):.1%} of data)")
print(zero['recommendation'].value_counts(normalize=True).round(3))
print(f"\nOverall base rate of 'Recommended': {(df['recommendation']=='Recommended').mean():.1%}")

Zero-hour reviews: 337 (2.3% of data)
recommendation
Not Recommended    0.727
Recommended        0.273
Name: proportion, dtype: float64

Overall base rate of 'Recommended': 71.1%


Zero-hour reviews are ~73% **negative** versus a ~29% negative base rate — a 2.5x
over-representation. These look like refund-and-review or protest reviews, i.e. **signal about
purchase regret, not measurement noise**. They are therefore flagged (so playtime-based metrics
can exclude them) but retained, and can be surfaced later as their own indicator.

## 4. Derived Features

Columns every downstream stage needs, computed once here:

- `rec` — recommendation as 1/0 (model-ready target/metric)
- `review_len_chars`, `review_len_words` — effort proxies (a 400-word essay ≠ "ok")
- `log_hours` — `log1p` of hours; the raw distribution is too right-skewed for means/linear use
- `year`, `month`, `quarter` — for time-series aggregation (trends, review-bomb detection)
- `engagement_votes` — helpful + funny (community interaction with the review)
- `confidence_tier` — per-game reliability tier based on review volume

In [13]:
df['rec'] = (df['recommendation'] == 'Recommended').astype(int)
df['review_len_chars'] = s.str.len()
df['review_len_words'] = s.str.split().str.len()
df['log_hours'] = np.log1p(df['hour_played'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.to_period('M').astype(str)
df['quarter'] = df['date'].dt.to_period('Q').astype(str)
df['engagement_votes'] = df['helpful'] + df['funny']

# Per-game confidence tier: statistics from <50 reviews are anecdote, not evidence
game_counts = df.groupby('title_clean')['rec'].transform('size')
df['confidence_tier'] = pd.cut(game_counts, bins=[0, 50, 150, np.inf],
                               labels=['Low', 'Medium', 'High'])

df[['title_clean', 'rec', 'hour_played', 'log_hours',
    'review_len_words', 'confidence_tier']].head(5)

,title_clean,rec,hour_played,log_hours,review_len_words,confidence_tier
0,Rust,1,40,3.713572,5,High
1,Euro Truck Simulator 2,1,20,3.044522,2,High
2,Rust,1,292,5.680173,3,High
3,Dead by Daylight,1,401,5.996452,1,High
4,Grand Theft Auto V,0,125,4.836282,76,High


## 5. Pipeline Eligibility Masks

Defines which games are eligible for each stage of the pipeline. This ensures all notebooks use the same filtering logic.

In [14]:
df['use_engagement'] = True

# NLP pipeline: needs genuine prose in English
df['use_nlp'] = ~(df['flag_null_review'] | df['flag_no_words'] |
                  df['flag_ascii_art'] | df['flag_non_english'])

# Playtime-based metrics: exclude zero-hour rows (no playtime to measure)
df['use_playtime'] = ~df['flag_zero_hours']

print("Retention by pipeline:")
for col in ['use_engagement', 'use_nlp', 'use_playtime']:
    print(f"  {col:<16} {df[col].sum():>6,} rows  ({df[col].mean():.1%})")

Retention by pipeline:
  use_engagement   14,617 rows  (100.0%)
  use_nlp          14,477 rows  (99.0%)
  use_playtime     14,280 rows  (97.7%)


## 6. Save to a new file

In [15]:
import os
os.makedirs('data', exist_ok=True)
OUT_PATH = 'data/reviews_prepared.csv'
df.to_csv(OUT_PATH, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_PATH}")

Saved 14,617 rows x 30 cols -> data/reviews_prepared.csv
